# Lab 2: Analysis Notebook — Framingham Heart Study

**Course:** PUBH 4201 — Applied Computing for Health Data Science
**Field track:** Epidemiology & Population Health
**Dataset:** Framingham Heart Study (teaching subset), loaded directly from its public source at runtime.

**Question:** How does 10-year coronary heart disease (CHD) risk differ across age groups and
current-smoking status, and which risk factors are most strongly associated with 10-year CHD
in a multivariable model?

> **AI assistance note:** I used Claude to help scaffold this notebook's structure, draft the
> `pd.cut()` age-bucketing logic, and explain how to convert logistic regression coefficients
> into odds ratios. All code was reviewed, run, and adjusted by me. See `AI_USAGE.md` in the
> repo root for the full log.

## 1. Load the data

Loaded directly from its original public source — no manual download/upload.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/TarekDib03/Analytics/master/Week3%20-%20Logistic%20Regression/Data/framingham.csv"
df = pd.read_csv(DATA_URL)

print(df.shape)
df.head()

## 2. Inspect structure and missingness

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

## 3. Non-trivial transformation

We:
1. Drop rows with any missing values (complete-case analysis — a limitation noted below).
2. Bucket `age` into four clinically meaningful groups.
3. Recode `currentSmoker` into a readable label.
4. Summarize CHD prevalence by age group × smoking status.

In [ ]:
df_clean = df.dropna().copy()
print(f"Rows before dropna: {len(df)}, after: {len(df_clean)}")

bins = [0, 40, 50, 60, 120]
labels = ["<=40", "41-50", "51-60", "61+"]
df_clean["age_group"] = pd.cut(df_clean["age"], bins=bins, labels=labels)
df_clean["smoker_status"] = df_clean["currentSmoker"].map({0: "Non-smoker", 1: "Current smoker"})

summary = (
    df_clean.groupby(["age_group", "smoker_status"], observed=True)["TenYearCHD"]
    .agg(n="size", chd_cases="sum", chd_rate="mean")
    .reset_index()
)
summary["chd_rate_pct"] = (summary["chd_rate"] * 100).round(1)
summary

## 4. Multivariable logistic regression (odds ratios)

Beyond the descriptive cross-tab above, we fit a logistic regression predicting `TenYearCHD`
from a small set of established risk factors, then exponentiate the coefficients to get odds
ratios (OR). An OR > 1 means the factor is associated with *higher* odds of 10-year CHD, holding
the other variables in the model constant.

In [ ]:
from sklearn.linear_model import LogisticRegression

features = ["age", "sysBP", "totChol", "currentSmoker", "diabetes"]
X = df_clean[features]
y = df_clean["TenYearCHD"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

odds_ratios = pd.Series(np.exp(model.coef_[0]), index=features, name="odds_ratio").sort_values(ascending=False)
odds_ratios

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

pivot = summary.pivot(index="age_group", columns="smoker_status", values="chd_rate_pct")
pivot.plot(kind="bar", ax=axes[0])
axes[0].set_title("10-year CHD rate by age group and smoking status")
axes[0].set_ylabel("CHD rate (%)")
axes[0].set_xlabel("Age group")
axes[0].legend(title="")
axes[0].tick_params(axis="x", rotation=0)

odds_ratios.plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].axvline(1.0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Odds ratios for 10-year CHD\n(multivariable logistic regression)")
axes[1].set_xlabel("Odds ratio")

plt.tight_layout()
plt.savefig("../rendered/framingham_figures.png", dpi=150)
plt.show()

## 6. Interpretation

Observed 10-year CHD prevalence rose steadily with age in both smoking groups, and within every
age band current smokers had a higher (or comparable) CHD rate than non-smokers, consistent with
smoking as a known cardiovascular risk factor. In the multivariable model, age and diabetes carried
the largest odds ratios, meaning each was associated with meaningfully higher odds of a 10-year CHD
event after adjusting for the other factors, while systolic blood pressure and cholesterol had more
modest effects at these ranges. These are unadjusted-for-confounding, observational associations
from a teaching subset — they describe risk patterns in this sample and should not be read as
causal effect estimates for any individual patient.

## AI assistance log (summary)

- Used Claude to help outline the notebook structure (load → clean → transform → model → visualize → interpret).
- Used Claude to draft the `pd.cut()` age-bucketing code and the multi-index `groupby().agg()` summary table.
- Used Claude to explain how `np.exp()` on logistic regression coefficients yields odds ratios, and to review the two-panel `matplotlib` figure code.
- All analytic choices (which features to include, which age cutpoints to use, the interpretation text) are my own.
- Full session-level detail is in `AI_USAGE.md` at the repo root.